# Paper QA：章节内递归切分（arXiv HTML）

输入是固定版本的 78 篇论文的 arXiv HTML 快照（`data/corpus_78`，由 `corpus_80` 去掉 Adam 和 PPO 两篇得到，
这两篇的 HTML 只是嵌入 PDF 的空壳，没有正文）；输出是本地 JSONL chunks、解析缓存和质量报告。
Notebook 调用 `scripts/chunk_papers.py`，与命令行使用同一套实现。

在 VS Code/Jupyter 中选择项目 `.venv` 的 Python 内核，然后依次运行所有单元格。
如使用新环境，先安装 `requirements-chunking.txt` 中的依赖。

流程：校验 HTML 快照哈希 → 按标签提取章节/段落 → 合并过短小节 → 小节内递归切分 → 来源与长度校验 → 保存。
默认：正文最多 512 tokens，重叠最多 64 tokens；排除参考文献和致谢，保留摘要与附录。
章节、参考文献和阅读顺序来自 HTML 标签，不再靠字体和版面猜测。

In [1]:
from pathlib import Path
import dataclasses
import importlib.util
import json
import sys
from html import escape
from IPython.display import HTML, display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts" / "chunk_papers.py").exists()
             and (p / "data" / "corpus_78" / "manifest.json").exists()), None)
if ROOT is None:
    raise RuntimeError("请从 Paper_QA_RAG 项目根目录或 notebooks 目录运行此 notebook。")
missing = [name for name in ["bs4", "lxml", "tokenizers", "huggingface_hub"] if importlib.util.find_spec(name) is None]
if missing:
    raise ModuleNotFoundError(f"缺少依赖 {missing}。请在当前内核运行 %pip install -r \"{ROOT / 'requirements-chunking.txt'}\"。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.chunk_papers import ChunkingConfig, run_pipeline, read_jsonl, get_tokenizer, token_count
print("项目目录:", ROOT)
print("当前 Python:", sys.executable)

项目目录: C:\Users\boyue\Desktop\LLM projects\Paper_QA_RAG
当前 Python: C:\Users\boyue\Desktop\LLM projects\Paper_QA_RAG\.venv\Scripts\python.exe


## 配置

`max_tokens` 是 chunk 正文的 token 上限，`overlap_tokens` 是同一小节内相邻 chunks 的最大重叠。
`min_section_tokens=100`：短于 100 tokens 的小节会并入同一一级章节下相邻的小节（先向后并，末尾的向前并），
被并入的小节标题会作为一行标题写进正文，不会丢失。这样不会出现只有一两句话的孤立 chunk。
tokenizer 使用 embedding 模型 `BAAI/bge-m3`（XLM-R SentencePiece）的 `tokenizer.json`，
因此 token 数与 embedding 实际输入一致；该模型输入上限为 8192 tokens，512 的 chunk 另加 2 个特殊 token 仍远低于上限。
首次运行需联网下载 tokenizer 文件，之后缓存在项目 `.cache/hf/`。

将 `LIMIT` 改成 `3` 可先预览三篇，设为 `None` 处理全部 78 篇。
改变配置或选择不同论文子集会生成不同的结果目录；解析缓存可以复用。

In [2]:
CONFIG = ChunkingConfig(
    max_tokens=512,
    overlap_tokens=64,
    tokenizer_name="BAAI/bge-m3",
    include_references=False,
    include_front_matter=False,
    source_format="html",
    min_section_tokens=100,
)
MANIFEST = ROOT / "data" / "corpus_78" / "manifest.json"
OUTPUT_ROOT = ROOT / "data" / "processed" / "corpus_78"
LIMIT = None
ONLY_PAPERS = None  # 例如 ["arxiv_1706.03762"]
FORCE_REPARSE = False
print(json.dumps(dataclasses.asdict(CONFIG), indent=2, ensure_ascii=False))

{
  "max_tokens": 512,
  "overlap_tokens": 64,
  "tokenizer_name": "BAAI/bge-m3",
  "include_references": false,
  "include_front_matter": false,
  "source_format": "html",
  "min_section_tokens": 100
}


## 运行并保存

章节来自 `<section>` 嵌套，参考文献来自 `ltx_bibliography`，公式保留 LaTeX（`$...$`），
表格按行展开为 `cell | cell`，图片只保留图注。转换错误元素（`ltx_ERROR`）会被跳过并计数。
同一小节中先尝试合并段落，超长段落再按句子、空白边界和 Unicode 字符边界细分。
正文始终取自规范化 section 文本的连续范围，切分阶段不会用 LLM 改写文本。

In [3]:
def show_progress(message):
    entry = json.loads(message)
    if entry["paper"] == 1 or entry["paper"] % 10 == 0 or entry["paper"] == entry["total"] or entry["status"] == "failed":
        print(f"{entry['paper']}/{entry['total']}: {entry['paper_id']}  {entry['status']}  chunks={entry['chunks']}")

result = run_pipeline(
    root=ROOT, manifest_path=MANIFEST, output_root=OUTPUT_ROOT,
    config=CONFIG, limit=LIMIT, paper_ids=ONLY_PAPERS,
    force_reparse=FORCE_REPARSE, progress=show_progress,
)
RUN_DIR = Path(result["output_dir"])
assert result["failed_papers"] == 0, "有论文处理失败，请查看 quality_report.json 后重跑。"

C:\Users\boyue\Desktop\LLM projects\Paper_QA_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1/78: arxiv_1706.03762  complete  chunks=29


10/78: arxiv_2203.15556  complete  chunks=69


20/78: arxiv_2412.01951  complete  chunks=217


30/78: arxiv_2601.18734  complete  chunks=45


40/78: arxiv_2306.09194  complete  chunks=93


50/78: arxiv_2302.06675  complete  chunks=64


60/78: arxiv_2302.04761  complete  chunks=48


70/78: arxiv_1508.07909  complete  chunks=32


78/78: arxiv_2602.02710  complete  chunks=110


In [4]:
metrics = {
    "处理论文数": result["completed_papers"],
    "chunks 数": result["total_chunks"],
    "待抽查论文数": result["papers_needing_review"],
    "最小 tokens": result["token_statistics"]["min"],
    "中位 tokens": result["token_statistics"]["median"],
    "最大 tokens": result["token_statistics"]["max"],
    "少于 64 tokens 的 chunks": result["token_statistics"]["chunks_under_64_tokens"],
}
rows = "".join(f"<tr><td style='padding:6px 18px'>{escape(str(k))}</td><td style='padding:6px 18px'>{escape(str(v))}</td></tr>" for k,v in metrics.items())
display(HTML("<table>" + rows + "</table>"))
print("输出目录:", RUN_DIR)

处理论文数,78
chunks 数,5755
待抽查论文数,7
最小 tokens,40
中位 tokens,412
最大 tokens,512
少于 64 tokens 的 chunks,12


输出目录: C:\Users\boyue\Desktop\LLM projects\Paper_QA_RAG\data\processed\corpus_78\chunks\section_recursive_512_64_e14374fa6d


## 检查章节识别

以下展示首篇论文解析出的章节树和各节正文长度（合并短小节之前的原始解析）。`kind=references` 的文本仍保留在解析缓存中，
只是默认不进入检索 chunks。每个 chunk 的字符偏移都以对应 section 的规范化文本为基准。

In [5]:
chunks = read_jsonl(RUN_DIR / "chunks.jsonl")
reports = read_jsonl(RUN_DIR / "papers.jsonl")
first = reports[0]
parsed = json.loads(Path(first["parsed_json"]).read_text(encoding="utf-8"))
for section in parsed["sections"][:35]:
    print("  " * max(0, section["level"] - 1) + section["title"],
          f"[{section['kind']}; chars={len(section['text'])}]")
print("\n本篇合并掉的短小节数:", first["merged_section_count"])

Abstract [body; chars=1139]
1 Introduction [body; chars=1918]
2 Background [body; chars=1816]
3 Model Architecture [body; chars=782]
  3.1 Encoder and Decoder Stacks [body; chars=1361]
  3.2 Attention [body; chars=336]
    3.2.1 Scaled Dot-Product Attention [body; chars=1871]
    3.2.2 Multi-Head Attention [body; chars=1657]
    3.2.3 Applications of Attention in our Model [body; chars=1179]
  3.3 Position-wise Feed-Forward Networks [body; chars=641]
  3.4 Embeddings and Softmax [body; chars=526]
  3.5 Positional Encoding [body; chars=1485]
4 Why Self-Attention [body; chars=3867]
5 Training [body; chars=58]
  5.1 Training Data and Batching [body; chars=590]
  5.2 Hardware and Schedule [body; chars=398]
  5.3 Optimizer [body; chars=502]
  5.4 Regularization [body; chars=568]
6 Results [body; chars=0]
  6.1 Machine Translation [body; chars=2726]
  6.2 Model Variations [body; chars=2706]
  6.3 English Constituency Parsing [body; chars=2463]
7 Conclusion [body; chars=1220]
References [refe

## 查看一个 chunk 及其来源

稳定的 `paragraph_id` 和 HTML 元素 ID（`element_id`）用于定位原文；`chunk_id` 还包含切分配置和字符范围。
评测集的证据以逐字摘录为准，再对齐到 chunk；不要只绑定 chunk ID。

In [6]:
sample = next((c for c in chunks if "Scaled Dot-Product" in c["section_title"]), chunks[0])
print(json.dumps({k: sample[k] for k in [
    "chunk_id", "paper_id", "version", "section_path", "token_count", "content_types",
    "char_start", "char_end", "source_format", "source_url"
]}, ensure_ascii=False, indent=2))
print("\n正文:\n" + sample["text"])
print("\n来源段落范围:\n" + json.dumps(sample["source_spans"], ensure_ascii=False, indent=2))

{
  "chunk_id": "chunk_cf4c20caebebb66002b1f5d7",
  "paper_id": "arxiv_1706.03762",
  "version": "v7",
  "section_path": [
    "Abstract"
  ],
  "token_count": 254,
  "content_types": [
    "text"
  ],
  "char_start": 0,
  "char_end": 1139,
  "source_format": "html",
  "source_url": "https://arxiv.org/html/1706.03762v7"
}

正文:
The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existi

## 输出检查与质量报告

程序已逐篇验证 token 上限、段落/章节字符范围、正文覆盖、排除规则和 chunk ID 唯一性。
这些检查不能证明 HTML 转换在语义上完全正确；`latex_conversion_errors_in_body` 表示该论文的 HTML 含有 LaTeXML 转换错误，
应抽查公式和表格密集的部分。图像未被读取，表格只保留文字。

In [7]:
encoding = get_tokenizer(CONFIG.tokenizer_name, ROOT)
assert len(chunks) == len({c["chunk_id"] for c in chunks})
assert all(0 < token_count(encoding, c["text"]) <= CONFIG.max_tokens for c in chunks)
assert all(c["source_spans"] and all(s["element_id"] for s in c["source_spans"]) for c in chunks)
if not CONFIG.include_references:
    assert all(c["section_kind"] != "references" for c in chunks)
quality = json.loads((RUN_DIR / "quality_report.json").read_text(encoding="utf-8"))
print("chunk 完整性检查通过。")
print("需要人工抽查的论文:")
for paper in quality["needs_review"]:
    print(paper["paper_id"], paper["quality_flags"], "错误元素数:", paper["latex_error_elements"])
if not quality["needs_review"]:
    print("本轮没有触发启发式质量标记；仍建议抽查。")

chunk 完整性检查通过。
需要人工抽查的论文:
arxiv_2405.21060 ['latex_conversion_errors_in_body'] 错误元素数: 1
arxiv_2412.01951 ['latex_conversion_errors_in_body'] 错误元素数: 54
arxiv_2408.03314 ['latex_conversion_errors_in_body'] 错误元素数: 1
arxiv_2601.18734 ['latex_conversion_errors_in_body'] 错误元素数: 9
arxiv_2501.00656 ['latex_conversion_errors_in_body'] 错误元素数: 54
arxiv_1508.07909 ['latex_conversion_errors_in_body'] 错误元素数: 1
arxiv_2503.14476 ['latex_conversion_errors_in_body'] 错误元素数: 2


## 本地目录

- `data/corpus_78/html/`：固定版本的 arXiv HTML 快照，`manifest.json` 记录 URL 和 SHA-256。
- `data/corpus_80/`：原始 80 篇清单和 PDF，保持不变。
- `data/processed/corpus_78/parsed/<parser_version>/`：每篇的章节/段落 JSON，以及方便抽查的 TXT。
- `data/processed/corpus_78/chunks/<run_id>/chunks.jsonl`：全部论文的 chunks，每行一条 JSON。
- 同目录下 `by_paper/`：按论文保存的 chunks；`summary.json`、`config.json`、`papers.jsonl` 和 `quality_report.json` 记录配置与质量。

后续 embedding 和 retrieval 直接读取 `chunks.jsonl`（也可读 `data/processed/corpus_78/latest_run.json` 找到最新一次）。
如果给正文加上论文标题、章节路径等前缀，需要按 embedding 模型的 tokenizer 重新计算完整输入长度。
此 notebook 不调用 embedding 或生成 API。